# 🔍 Session 1: Grundlagen von Retrieval-Augmented Generation (RAG)

**PIRAGE Trainingskurs — Tag 1, Session 1**

> *Kein API-Key erforderlich — läuft vollständig im Browser via Google Colab*

---

## 🎯 Lernziele dieser Session

Nach dieser Session verstehst du:
- Was RAG ist und **warum** es so wichtig ist
- Die grundlegenden Herausforderungen beim Aufbau von Retrieval-Pipelines
- Das **PIRAGE-Framework** als Leitfaden durch den gesamten Kurs
- Den Unterschied zwischen reinen LLMs und RAG-Systemen

---

## 🧠 Was ist RAG?

**Retrieval-Augmented Generation (RAG)** kombiniert zwei Komponenten:

1. **Retrieval** — Relevante Informationen aus einer Wissensbasis abrufen
2. **Generation** — Ein Sprachmodell (LLM) nutzt diese Informationen für präzise Antworten

```
┌─────────────────────────────────────────────────────────────┐
│                    RAG-Architektur                          │
│                                                             │
│  Frage ──→ [Retrieval] ──→ relevante Dokumente ──┐          │
│                                                  ↓          │
│                                    [LLM + Kontext] ──→ Antwort │
└─────────────────────────────────────────────────────────────┘
```

### Das Problem ohne RAG

Stell dir vor, du fragst ein LLM: *"Was steht in unserem Unternehmenshandbuch auf Seite 42?"*

Das LLM kann diese Frage **nicht beantworten**, weil:
- Es nur auf Trainingsdaten bis zu einem Stichtag trainiert wurde
- Es keinen Zugriff auf private Dokumente hat
- Es bei fehlendem Wissen **halluziniert** (falsche Antworten erfindet)

**RAG löst genau dieses Problem!**

## 🏛️ Das PIRAGE-Framework

PIRAGE ist unser Leitfaden durch den gesamten RAG-Lebenszyklus:

| Phase | Buchstabe | Beschreibung |
|-------|-----------|-------------|
| **P**arse | P | Dokumente einlesen: PDF, Word, CSV, Web → reiner Text |
| **I**ndex | I | Text in Vektoren umwandeln und durchsuchbar machen |
| **R**etrieval | R | Die relevantesten Textpassagen zur Anfrage finden |
| **A**ugmented **G**eneration | AG | LLM mit Kontext versorgen → präzise Antwort |
| **E**valuation | E | Qualität messen, Fehler debuggen, optimieren |

```
 P ─── I ─── R ─── AG ─── E
 │     │     │      │      │
 │     │     │      │      └── Qualitätsmessung & Feedback
 │     │     │      └───────── LLM + Kontext → Antwort
 │     │     └──────────────── Relevante Chunks finden
 │     └────────────────────── Vektorindex aufbauen
 └──────────────────────────── Dokumente aufbereiten
```

In diesem Kurs durchlaufen wir jeden Schritt — von Naive RAG bis zu fortgeschrittenen Agentic-Ansätzen.

## 📊 Brainstorming: Der moderne AI Researcher

### 🔖 Übung 1 — Warum brauchen Unternehmen RAG?

Betrachte folgende typische Stellenbeschreibung für einen **AI Research Engineer**:

---
*"Gesucht: AI Research Engineer für die Entwicklung interner Wissensmanagementsysteme.*
*Aufgaben: Implementierung von Dokumentensuche über 50.000+ interne Dokumente,*
*Integration von LLMs mit bestehenden Datenbanken, Sicherstellung von Datenschutz.*
*Anforderungen: Erfahrung mit Vektordatenbanken, Embedding-Modellen, LangChain oder ähnlichen Frameworks."*

---

**Diskutiere:** Welche RAG-Herausforderungen siehst du in dieser Rolle?

Typische Herausforderungen:

1. **Skalierung** — 50.000 Dokumente effizient durchsuchen (Latenz!)
2. **Aktualität** — Neue Dokumente müssen in den Index aufgenommen werden
3. **Präzision vs. Recall** — Zu viele irrelevante Treffer vs. wichtige Treffer verpassen
4. **Datenschutz** — Sensible Daten dürfen nicht an externe APIs gesendet werden
5. **Halluzinationen** — Das LLM darf keine Fakten erfinden

## 💻 Interaktive Demo — RAG vs. Kein RAG

Jetzt sehen wir den Unterschied zwischen einem LLM **ohne** und **mit** Kontext:

Wir bauen ein Mini-Beispiel **ohne echtes LLM** — nur mit dem Retrieval-Teil, um den Kernmechanismus zu zeigen.

In [ ]:
# Installation der benötigten Pakete
# Läuft in Google Colab ohne zusätzliche Konfiguration
%pip install -q sentence-transformers faiss-cpu numpy

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

print('✅ Pakete geladen')
print('📦 sentence-transformers: Erstellt Embeddings aus Text')
print('📦 faiss: Schnelle Vektorsuche von Meta AI')

In [ ]:
# Unsere kleine Wissensbasis — typische Unternehmens-Dokumente
wissensbasis = [
    {
        "id": 1,
        "titel": "Urlaubsregelung 2024",
        "inhalt": "Mitarbeiter haben Anspruch auf 30 Tage Urlaub pro Jahr. Urlaub muss 2 Wochen im Voraus beantragt werden. Resturlaub verfällt am 31. März des Folgejahres."
    },
    {
        "id": 2,
        "titel": "Homeoffice-Policy",
        "inhalt": "Mitarbeiter können bis zu 3 Tage pro Woche im Homeoffice arbeiten. An Montagen und Freitagen ist Anwesenheit im Büro empfohlen. Für Homeoffice wird ein geeigneter Arbeitsplatz vorausgesetzt."
    },
    {
        "id": 3,
        "titel": "IT-Sicherheitsrichtlinien",
        "inhalt": "Passwörter müssen mindestens 12 Zeichen lang sein und alle 90 Tage geändert werden. Externe USB-Sticks sind nicht erlaubt. VPN muss bei Remote-Arbeit aktiviert sein."
    },
    {
        "id": 4,
        "titel": "Reisekostenabrechnung",
        "inhalt": "Dienstreisen müssen vorab genehmigt werden. Hotel-Budget: max. 120€ pro Nacht. Bahnfahrten werden vollständig erstattet. Flüge ab 5 Stunden Reisezeit erlaubt."
    },
    {
        "id": 5,
        "titel": "Onboarding-Prozess",
        "inhalt": "Neue Mitarbeiter erhalten in der ersten Woche eine Einführung in alle Teams. Buddy-System: Jeder neue Mitarbeiter bekommt einen erfahrenen Kollegen als Ansprechpartner für 3 Monate."
    }
]

print(f'📚 Wissensbasis: {len(wissensbasis)} Dokumente geladen')
for doc in wissensbasis:
    print(f'  [{doc["id"]}] {doc["titel"]}')

In [ ]:
# Embedding-Modell laden (all-MiniLM-L6-v2 — ~80MB, läuft auf Colab Free Tier)
print('🔄 Lade Embedding-Modell... (dauert ~30 Sekunden beim ersten Mal)')
modell = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Modell geladen!')

# Texte aus der Wissensbasis extrahieren
texte = [doc['inhalt'] for doc in wissensbasis]

# Embeddings erstellen — das ist der 'I' (Index) Schritt in PIRAGE
print('\n🔄 Erstelle Embeddings...')
embeddings = modell.encode(texte, show_progress_bar=True)
print(f'\n✅ Embeddings erstellt!')
print(f'📐 Embedding-Dimensionen: {embeddings.shape}')
print(f'   → {len(texte)} Dokumente × {embeddings.shape[1]} Dimensionen')

In [ ]:
# FAISS-Index aufbauen — schnelle Vektorsuche
dimension = embeddings.shape[1]  # 384 für all-MiniLM-L6-v2
index = faiss.IndexFlatL2(dimension)  # L2 = Euklidischer Abstand

# Embeddings zum Index hinzufügen
index.add(embeddings.astype('float32'))

print(f'✅ FAISS-Index aufgebaut')
print(f'📊 Gespeicherte Vektoren: {index.ntotal}')
print(f'🔍 Index-Typ: IndexFlatL2 (exakte Suche, geeignet für kleine Datenmengen)')

In [ ]:
def retrieval(frage: str, k: int = 2) -> list:
    """Findet die k relevantesten Dokumente für eine Frage."""
    # 'R' (Retrieval) Schritt in PIRAGE:
    # 1. Frage als Embedding kodieren
    frage_embedding = modell.encode([frage])

    # 2. Nächste Nachbarn im Vektorraum suchen
    distanzen, indizes = index.search(frage_embedding.astype('float32'), k)

    # 3. Ergebnisse aufbereiten
    ergebnisse = []
    for i, (abstand, idx) in enumerate(zip(distanzen[0], indizes[0])):
        doc = wissensbasis[idx]
        ergebnisse.append({
            'rang': i + 1,
            'dokument': doc,
            'aehnlichkeit': 1 / (1 + abstand)  # Abstand → Ähnlichkeit umrechnen
        })

    return ergebnisse


def zeige_ergebnisse(frage: str, k: int = 2):
    """Zeigt Retrieval-Ergebnisse für eine Frage."""
    print(f'\n❓ Frage: "{frage}"')
    print('=' * 60)

    ergebnisse = retrieval(frage, k)

    print(f'📎 Top-{k} relevante Dokumente:')
    for r in ergebnisse:
        print(f"\n  [{r['rang']}] {r['dokument']['titel']} (Ähnlichkeit: {r['aehnlichkeit']:.3f})")
        print(f"      {r['dokument']['inhalt'][:100]}...")

    print('\n💡 Prompt für LLM würde lauten:')
    kontext = '\n'.join([r['dokument']['inhalt'] for r in ergebnisse])
    print(f'   Kontext: {kontext[:150]}...')
    print(f'   Frage: {frage}')


print('✅ Retrieval-Funktion definiert')

In [ ]:
# Demo: Verschiedene Fragen testen
fragen = [
    "Wie viele Tage Urlaub habe ich?",
    "Darf ich von zuhause arbeiten?",
    "Wie sicher muss mein Passwort sein?",
    "Wie funktioniert die Einarbeitung neuer Mitarbeiter?"
]

for frage in fragen:
    zeige_ergebnisse(frage, k=1)
    print()

## 🔬 Übung 2 — Ähnlichkeit selbst erfahren

Probiere verschiedene Formulierungen derselben Frage aus und beobachte, wie das System reagiert:

In [ ]:
# Vergleich: verschiedene Formulierungen
varianten = [
    "Wie viele Urlaubstage stehen mir zu?",
    "Vacation days allowance",              # Englisch funktioniert auch!
    "Wann verfällt nicht genommener Urlaub?",
    "Kann ich Urlaub auf nächstes Jahr übertragen?"
]

print('🔍 Wie robust ist unser Retrieval?\n')
for v in varianten:
    zeige_ergebnisse(v, k=1)

## 🧩 Was haben wir heute gelernt?

### Zusammenfassung Session 1

| Konzept | Kernaussage |
|---------|-------------|
| **RAG** | Kombiniert Retrieval mit LLM-Generierung für faktenbasierte Antworten |
| **Embedding** | Text wird in Zahlen-Vektoren umgewandelt, die semantische Bedeutung erfassen |
| **Vektorähnlichkeit** | Ähnliche Texte haben ähnliche Vektoren — das ist die Basis von RAG |
| **FAISS** | Effiziente Bibliothek für die Suche in großen Vektordatenbanken |
| **PIRAGE** | Unser Framework: Parse → Index → Retrieval → Augmented Generation → Evaluation |

### Was kommt in Session 2+3?

Wir bauen die **vollständige Naive RAG-Pipeline** inkl.:
- Dokumente einlesen & Chunking-Strategien
- Vollständiger RAG-Workflow mit Prompt-Konstruktion
- Vergleich verschiedener Chunking-Ansätze

---

## 🏋️ Bonusübung (optional)

Erweitere die Wissensbasis um 2 eigene Dokumente und teste, ob das Retrieval funktioniert:

```python
wissensbasis.append({
    "id": 6,
    "titel": "Dein Dokument",
    "inhalt": "Dein Text hier..."
})
# Dann: Embeddings neu erstellen und Index neu aufbauen!
```

---
*PIRAGE RAG Kurs — Session 1 | 🦃 [Copilotclaw/trainingdemo](https://github.com/Copilotclaw/trainingdemo)*